# 09 — Claude Haiku Grading Evaluator

Exact Python port of the Chrome extension grading pipeline for offline evaluation.

Replicates **every step** of `claudeVision.ts` + `cvDetectors.ts`:
1. CV detectors — Laplacian blur, glare fraction, brightness (same constants as production)
2. Image resize — longest edge ≤ 1024 px, JPEG q85
3. Prompt injection — CV measurements prepended verbatim
4. Claude Haiku call — same model, max_tokens, system/user prompts
5. Distribution normalisation

Use this to:
- Test new prompts before deploying
- Evaluate consistency across multiple images of the same card
- Build a labelled dataset for accuracy benchmarking
- Debug why a particular card got an unexpected grade

In [ ]:
# Run once, then comment out
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'anthropic', 'Pillow', 'numpy', 'opencv-python', 'requests',
    'matplotlib', 'pandas', 'tqdm'])
print('✅ Dependencies ready')

In [ ]:
import os, io, re, json, base64, textwrap, warnings
from pathlib import Path

import numpy as np
import cv2
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import anthropic
from tqdm.auto import tqdm
from IPython.display import display as ipy_display, HTML

warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi':       120,
    'axes.facecolor':   '#161b22',
    'figure.facecolor': '#0d1117',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('✅ Imports OK')

## Configuration

Set your Anthropic API key. The notebook reads `ANTHROPIC_API_KEY` from the environment, or you can paste it directly.

In [ ]:
# ── API Key ───────────────────────────────────────────────────────────────────
API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not API_KEY:
    API_KEY = input('Paste your Anthropic API key: ').strip()
assert API_KEY, 'API key required'
print(f'✅ API key loaded ({API_KEY[:8]}...)')

## CV Detectors

Exact Python port of `cvDetectors.ts`. Same constants, same algorithm.

| Constant | Value | Meaning |
|---|---|---|
| `CANONICAL_W` | 384 | Resize width for CV analysis |
| `CANONICAL_H` | 544 | Resize height for CV analysis |
| `BLUR_THRESHOLD` | 40 | Laplacian variance below = blurry |
| `GLARE_THRESHOLD` | 0.05 | Fraction of pixels ≥ 245 above = significant glare |

In [ ]:
# ── Constants — must match cvDetectors.ts ─────────────────────────────────────
CANONICAL_W     = 384
CANONICAL_H     = 544
BLUR_THRESHOLD  = 40
GLARE_THRESHOLD = 0.05


def _laplacian_variance(gray: np.ndarray) -> float:
    """Laplacian variance using the same 5-point kernel as cvDetectors.ts.

    TypeScript iterates manually; cv2.Laplacian uses the identical
    cross-shaped kernel (0,1,0 / 1,-4,1 / 0,1,0).  Results match within
    floating-point rounding.
    """
    lap = cv2.Laplacian(gray.astype(np.float64), cv2.CV_64F)
    return float(lap.var())


def _analyse_image_array(img_rgb: np.ndarray) -> dict:
    """Run detectors on an already-loaded RGB numpy array."""
    # Resize to canonical (fill, not fit-inside — matches sharp 'fill' mode)
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H),
                           interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)

    total            = gray.size
    brightness_mean  = float(gray.mean())
    brightness_std   = float(gray.std())
    glare_count      = int(np.sum(gray >= 245))
    glare_fraction   = glare_count / total
    blur_score       = _laplacian_variance(gray)

    is_blurry = blur_score < BLUR_THRESHOLD
    has_glare = glare_fraction > GLARE_THRESHOLD

    cv_issues = []
    if is_blurry:
        cv_issues.append(
            f'blurry image (score {blur_score:.1f}, need \u2265{BLUR_THRESHOLD})'
            ' \u2014 reduce grade confidence')
    if has_glare:
        cv_issues.append(
            f'surface glare on {glare_fraction*100:.1f}% of card area'
            ' \u2014 may hide scratches or haze')
    if brightness_mean < 50:
        cv_issues.append('very dark image \u2014 poor lighting may obscure surface defects')
    elif brightness_mean > 220 and not has_glare:
        cv_issues.append('overexposed image \u2014 highlight detail may be lost')

    return {
        'blur_score':      round(blur_score, 1),
        'glare_fraction':  round(glare_fraction, 4),
        'brightness_mean': round(brightness_mean, 1),
        'brightness_std':  round(brightness_std, 1),
        'is_blurry':       is_blurry,
        'has_glare':       has_glare,
        'cv_issues':       cv_issues,
    }


def run_cv_detectors(image_urls: list[str]) -> dict | None:
    """Port of runCVDetectors(). Tries first 2 URLs, returns first success."""
    for url in image_urls[:2]:
        try:
            resp = requests.get(url, timeout=8)
            resp.raise_for_status()
            img = np.array(Image.open(io.BytesIO(resp.content)).convert('RGB'))
            result = _analyse_image_array(img)
            result['_source_url'] = url
            return result
        except Exception as exc:
            print(f'  CV: skipping {url[:70]}... ({exc})')
    return None


def run_cv_detectors_local(image_path: str) -> dict | None:
    """CV detectors on a local file path (for offline testing)."""
    try:
        img = np.array(Image.open(image_path).convert('RGB'))
        result = _analyse_image_array(img)
        result['_source_url'] = image_path
        return result
    except Exception as exc:
        print(f'  CV local: failed ({exc})')
        return None


def format_cv_section(cv: dict | None) -> str:
    """Port of formatCVSection(). Returns empty string if cv is None."""
    if cv is None:
        return ''
    issue_str = (
        '\n'.join(f'  \u2022 {s}' for s in cv['cv_issues'])
        if cv['cv_issues'] else '  \u2022 none detected'
    )
    blur_note  = ('\nNOTE: Image is blurry \u2014 lower confidence, flag in issues.'
                  if cv['is_blurry'] else '')
    glare_note = ('\nNOTE: Significant glare \u2014 surface haze or scratches may be hidden.'
                  if cv['has_glare'] else '')
    return (
        f'\u2500\u2500\u2500 CV MEASUREMENTS (pixel analysis, run before this prompt) \u2500\u2500\u2500\n'
        f'Blur score   : {cv["blur_score"]}  (threshold \u2265 {BLUR_THRESHOLD} = acceptably sharp)\n'
        f'Glare        : {cv["glare_fraction"]*100:.1f}% pixels overexposed  (\u2264 5% acceptable)\n'
        f'Brightness   : mean {cv["brightness_mean"]} / std {cv["brightness_std"]}  (0\u2013255 scale)\n'
        f'CV issues    :\n'
        f'{issue_str}{blur_note}{glare_note}\n\n'
    )


print('\u2705 CV detectors ready')

## Image Helpers

Port of `fetchResized()` from `claudeVision.ts` — downloads and resizes the primary image to ≤ 1024 px for fewer Claude input tokens.

In [ ]:
def fetch_resized(url: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Port of fetchResized() from claudeVision.ts.

    Downloads url, resizes longest edge to ≤ max_size (no upscaling),
    encodes as JPEG base64.  Returns None on any failure.
    """
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  resize failed: {exc}')
        return None


def load_local_image(path: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Same as fetch_resized but from a local file path."""
    try:
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  local load failed: {exc}')
        return None


def show_images(image_urls: list[str] | None = None,
                local_paths: list[str] | None = None,
                max_show: int = 6):
    """Display thumbnail grid of the test images."""
    sources = []
    if image_urls:
        for url in image_urls[:max_show]:
            try:
                resp = requests.get(url.replace('s-l1600', 's-l300'), timeout=6)
                sources.append(Image.open(io.BytesIO(resp.content)).convert('RGB'))
            except Exception:
                pass
    if local_paths:
        for p in local_paths[:max_show]:
            try:
                sources.append(Image.open(p).convert('RGB'))
            except Exception:
                pass

    if not sources:
        print('No images to display')
        return

    n = len(sources)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, img in zip(axes, sources):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'{n} image(s)', color='white', fontsize=11)
    plt.tight_layout()
    plt.show()


print('\u2705 Image helpers ready')

## Prompts

Copied verbatim from `src/lib/grading/claudeVision.ts`. **Edit these here to test prompt changes before deploying.**

In [ ]:
SYSTEM_PROMPT = """You are an expert PSA pre-grading assistant specializing in Pokémon trading cards.

Your task is to estimate the plausible PSA grade range supported by the visible evidence. You are not guaranteeing a final PSA grade.

Core principles:
- Base your assessment only on what is actually visible in the images.
- Never assume unseen areas are clean.
- If glare, blur, angle, cropping, sleeve reflections, compression, or low resolution hide details, reduce confidence and mention the limitation.
- When evidence is incomplete, prefer a wider grade range and lower confidence.
- Distinguish between: (1) visible defects and (2) unknowns caused by image limitations.

Evaluate in this order:
1. IMAGE CLASSIFICATION — label each image as "front", "back", or "other" based on visible content. The front shows the card artwork; the back shows the Pokémon logo / back design.
2. FRONT ANALYSIS — if a front image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR — whitening, fraying, rounding
   - Edges: top, bottom, left, right — nicks, chips, whitening
   - Surface: scratches, print lines, holo damage, staining, gloss loss
3. BACK ANALYSIS — if a back image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR
   - Edges: top, bottom, left, right
   - Surface: print quality, scratches, staining
4. COMBINED GRADE — derive the final grade_estimate and combined issues from the worst-case evidence across both sides.
5. CARD IDENTITY — identify name/set/year/number from the front image only.

PSA grade guidance:
- PSA 10 Gem Mint: near-perfect, centering ~55/45 or better on front, four sharp corners, no visible defects, full gloss
- PSA 9  Mint: slight visible wear allowed, minor issues possible, strong overall presentation
- PSA 8  NM-MT: light corner/edge wear, possible very light scratches or print issues
- PSA 7  NM: visible but not severe corner/edge wear, possible light scratches or minor print defects
- PSA 6  EX-MT and below: increasingly obvious wear, scratches, edge damage, staining, creasing, or major defects

Front-only rule (CRITICAL):
- If no back image is identifiable, set analysis_mode to "front_only"
- Set back_analysis.assessable to false and leave all back_analysis issue arrays empty
- Set grade_estimate.confidence to "low" regardless of front image quality
- Set grade_estimate.limiting_factor to "front_only"
- Add exactly this string to grading_decision.caveats: "Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."
- Do NOT speculate about the back condition

Other special instructions:
- Do not double-count the same defect across front and back.
- Vintage Pokémon cards (1999–2003) should be judged with awareness of era, but never use era to override visible evidence.
- Holo surface must be treated conservatively when glare or angle prevents reliable inspection.
- Do not overclaim PSA 9 or PSA 10 when image quality is limited.

Respond ONLY with one valid JSON object. No markdown, no prose outside JSON.

Every "issues" object must use exactly these keys: centering, corners, edges, surface, other.
Each maps to an array of strings. Use [] when nothing is visible for that category.

Allowed values:
- "analysis_mode": "front_only" or "front_back"
- "image_quality.status": "good", "partial", or "poor"
- "card_identity.confidence": "high", "medium", or "low"
- "grade_estimate.confidence": "high", "medium", or "low"
- "grade_estimate.limiting_factor": "front_only", "image_quality", "visible_damage", or null
- "grading_decision.gradable_candidate": "yes", "maybe", or "no"
- "front_analysis.assessable" / "back_analysis.assessable": true or false

If a card_identity field is uncertain, use null rather than guessing."""


USER_PROMPT = """Analyze this Pokémon card from the provided image set and return JSON with EXACTLY this structure.

Example when both front and back are present:
{
  "analysis_mode": "front_back",
  "card_identity": {
    "name": "Charizard", "set": "Base Set", "year": "1999", "number": "4", "confidence": "high"
  },
  "image_quality": {
    "status": "good", "warnings": [],
    "front_present": true, "back_present": true,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": true
  },
  "front_analysis": {
    "assessable": true,
    "centering": "58/42 L/R, 54/46 T/B",
    "issues": {
      "centering": ["Slightly left-heavy, approximately 58/42 L/R"],
      "corners":   ["Light whitening on top-right corner"],
      "edges":     [],
      "surface":   ["Faint holo scratches visible at angle"],
      "other":     []
    }
  },
  "back_analysis": {
    "assessable": true,
    "centering": "50/50 L/R, 51/49 T/B",
    "issues": {
      "centering": [],
      "corners":   [],
      "edges":     ["Minor whitening on bottom edge"],
      "surface":   [],
      "other":     []
    }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-8", "confidence": "high", "limiting_factor": "visible_damage",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.04,
      "6": 0.08, "7": 0.30, "8": 0.40, "9": 0.12, "10": 0.03
    }
  },
  "issues": {
    "centering": ["Slightly left-heavy front (58/42 L/R)"],
    "corners":   ["Light whitening on top-right corner (front)"],
    "edges":     ["Minor whitening on bottom edge (back)"],
    "surface":   ["Faint holo scratches visible at angle (front)"],
    "other":     []
  },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Card shows light wear consistent with PSA 7-8. Holo scratches are the primary concern.",
    "caveats": []
  }
}

Example when only the front is present (front_only):
{
  "analysis_mode": "front_only",
  "card_identity": { "name": null, "set": null, "year": null, "number": null, "confidence": "low" },
  "image_quality": {
    "status": "partial", "warnings": [], "front_present": true, "back_present": false,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": false
  },
  "front_analysis": {
    "assessable": true,
    "centering": "55/45 L/R, 53/47 T/B",
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "back_analysis": {
    "assessable": false,
    "centering": null,
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-9", "confidence": "low", "limiting_factor": "front_only",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.05,
      "6": 0.10, "7": 0.22, "8": 0.30, "9": 0.22, "10": 0.08
    }
  },
  "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Front appears reasonably clean but back is not available for assessment.",
    "caveats": ["Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."]
  }
}

Rules:
- Output exactly one JSON object. Use exactly the top-level keys shown above.
- front_analysis and back_analysis each have their own "issues" object for per-side defects.
- "issues" at the top level is the COMBINED worst-case from both sides. Tag each item with "(front)" or "(back)".
- Do not guess hidden defects. Only report what is actually visible.
- If the back is missing: set analysis_mode "front_only", back_analysis.assessable false, grade confidence "low", limiting_factor "front_only", and add the caveat string exactly as shown.
- If glare/blur/angle limits visibility, mention in image_quality.warnings and the relevant side analysis.
- Use a wider grade range and lower confidence when evidence is incomplete.
- Be conservative about PSA 9 and PSA 10 if surface or back visibility is limited.
- Distribution values must sum approximately to 1.00.
- confidence "high": top-2 PSA grades >60% of mass. "medium": 40–60%. "low": image too limited."""


print('✅ Prompts loaded')
print(f'  System prompt: {len(SYSTEM_PROMPT):,} chars')
print(f'  User prompt:   {len(USER_PROMPT):,} chars')

## Main Grading Function

Exact port of `gradeWithClaude()` from `claudeVision.ts`.

In [ ]:
def grade_with_claude(
    image_urls:   list[str] | None = None,
    local_paths:  list[str] | None = None,
    title:        str = '',
    api_key:      str = API_KEY,
    show_prompt:  bool = False,
    show_raw:     bool = False,
) -> dict:
    """Exact port of gradeWithClaude() + runCVDetectors() from claudeVision.ts.

    Args:
        image_urls:  eBay image URLs (s-l1600 preferred).  Primary image is
                     resized + sent as base64; extras sent as URL references.
        local_paths: Local JPEG/PNG paths.  All sent as base64.
                     Supply either image_urls or local_paths, not both.
        title:       Listing title injected as identity hint.
        api_key:     Anthropic API key.
        show_prompt: Print the exact prompt sent to Claude.
        show_raw:    Print the raw string Claude returned before JSON parsing.

    Returns:
        Parsed result dict with added _cv and _usage keys.
    """
    client = anthropic.Anthropic(api_key=api_key)

    # ── Step 1: CV detectors ──────────────────────────────────────────────────
    print('  [1/4] Running CV detectors...')
    if image_urls:
        cv = run_cv_detectors(image_urls)
    elif local_paths:
        cv = run_cv_detectors_local(local_paths[0])
    else:
        cv = None

    # ── Step 2: Fetch + resize primary image ──────────────────────────────────
    print('  [2/4] Fetching + resizing primary image...')
    resized = None
    if image_urls:
        resized = fetch_resized(image_urls[0])
    elif local_paths:
        resized = load_local_image(local_paths[0])

    # ── Step 3: Build image content blocks ───────────────────────────────────
    image_blocks = []

    if resized:
        image_blocks.append({
            'type': 'image',
            'source': {
                'type':       'base64',
                'media_type': resized['media_type'],
                'data':       resized['base64'],
            }
        })
    elif image_urls:
        image_blocks.append({
            'type':   'image',
            'source': {'type': 'url', 'url': image_urls[0]}
        })

    # Additional images: URLs sent directly; local paths sent as base64
    if image_urls:
        for url in image_urls[1:6]:
            image_blocks.append({
                'type':   'image',
                'source': {'type': 'url', 'url': url}
            })
    if local_paths:
        for path in local_paths[1:6]:
            extra = load_local_image(path)
            if extra:
                image_blocks.append({
                    'type': 'image',
                    'source': {
                        'type':       'base64',
                        'media_type': extra['media_type'],
                        'data':       extra['base64'],
                    }
                })

    # ── Step 4: Build prompt ─────────────────────────────────────────────────
    cv_section = format_cv_section(cv)
    title_hint = f'Listing title (use as identity hint, but trust the image over the title): "{title}"\n\n' if title else ''
    full_text  = f'{cv_section}{title_hint}{USER_PROMPT}'

    if show_prompt:
        print('\n' + '='*70)
        print('SYSTEM PROMPT:')
        print('='*70)
        print(SYSTEM_PROMPT)
        print('\n' + '='*70)
        print('USER TEXT BLOCK (prepended to images):')
        print('='*70)
        print(full_text)
        print('='*70 + '\n')

    # ── Step 5: Call Claude Haiku ────────────────────────────────────────────
    # max_tokens=2048: the front+back split schema is ~600-900 tokens output;
    # 1024 was too small and caused truncated JSON errors.
    print(f'  [3/4] Calling Claude Haiku ({len(image_blocks)} image block(s))...')
    response = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=[{
            'role': 'user',
            'content': [*image_blocks, {'type': 'text', 'text': full_text}]
        }]
    )

    raw = ''.join(
        block.text for block in response.content if block.type == 'text'
    ).strip()

    if show_raw:
        print('\n--- RAW RESPONSE ---')
        print(raw)
        print('--- END RAW ---\n')

    # Strip accidental markdown code fences (leading and trailing)
    json_str = re.sub(r'^```(?:json)?\s*\n?', '', raw, flags=re.IGNORECASE)
    json_str = re.sub(r'\n?```\s*$', '', json_str).strip()

    print('  [4/4] Parsing response...')
    try:
        result = json.loads(json_str)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'Claude returned non-JSON (output_tokens={response.usage.output_tokens}):\n{raw[:600]}'
        ) from exc

    # Normalise distribution to sum to 1.0
    dist  = result.get('grade_estimate', {}).get('distribution', {})
    total = sum(dist.values())
    if total > 0 and abs(total - 1) > 0.02:
        for k in dist:
            dist[k] = round(dist[k] / total, 4)

    # Attach metadata
    result['_cv']    = cv
    result['_usage'] = {
        'input_tokens':  response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
    }

    return result


print('✅ grade_with_claude() ready')

## Display Helpers

In [ ]:
GRADE_COLORS = {
    10: '#10b981', 9: '#34d399', 8: '#6366f1',
    7:  '#818cf8', 6: '#f97316', 5: '#fb923c',
    4:  '#ef4444', 3: '#dc2626', 2: '#b91c1c', 1: '#7f1d1d',
}

ISSUE_CATEGORY_LABELS = {
    'centering': 'Centering',
    'corners':   'Corners',
    'edges':     'Edges',
    'surface':   'Surface',
    'other':     'Other',
}


def _print_side_issues(issues: dict) -> bool:
    """Print per-side issues dict; returns True if any issues found."""
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'      {cat.upper()}')
            for item in items:
                print(f'        • {item}')
    return has_any


def print_result(result: dict, label: str = '') -> None:
    """Pretty-print a single grading result to stdout."""
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    grade  = result.get('grade_estimate', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    cv     = result.get('_cv')
    usage  = result.get('_usage', {})
    mode   = result.get('analysis_mode', 'front_only')

    sep = '=' * 65
    print(f'\n{sep}')
    if label:
        print(f'  {label}')
    print(sep)

    # Card identity
    name    = card.get('name') or '(unknown)'
    cset    = card.get('set') or '?'
    year    = card.get('year') or '?'
    num     = card.get('number') or '?'
    id_conf = card.get('confidence', '?')
    print(f'\n🃏  {name}  [{cset} {year} #{num}]')
    print(f'    ID confidence: {id_conf}   Mode: {mode}')

    # Image quality
    status = iq.get('status', '?').upper()
    print(f'\n📷  Image Quality: {status}')
    chip_defs = [
        ('front_present', 'Front'), ('back_present', 'Back'),
        ('centering_visible', 'Centering'), ('corners_visible', 'Corners'),
        ('edges_visible', 'Edges'),         ('surface_visible', 'Surface'),
    ]
    chips = '  '.join(f"{'✓' if iq.get(k) else '✗'} {lbl}" for k, lbl in chip_defs)
    print(f'    {chips}')
    for w in iq.get('warnings', []):
        print(f'    ⚠ {w}')

    # CV measurements
    if cv:
        bf = '🔴' if cv['is_blurry'] else '🟢'
        gf = '🔴' if cv['has_glare'] else '🟢'
        print(f'\n🔬  CV: blur {cv["blur_score"]:.1f}{bf}  '
              f'glare {cv["glare_fraction"]*100:.1f}%{gf}  '
              f'brightness {cv["brightness_mean"]:.0f}')

    # ── Front Analysis ────────────────────────────────────────────
    print(f'\n── Front Analysis ──────────────────────────────────────────')
    if front.get('assessable') is False:
        print('    ✗ Not assessable — no front image provided')
    else:
        centering = front.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        front_issues = front.get('issues', {})
        if not _print_side_issues({k: v for k, v in front_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Back Analysis ─────────────────────────────────────────────
    print(f'\n── Back Analysis ───────────────────────────────────────────')
    if back.get('assessable') is False:
        print('    ✗ Not assessable — no back image provided')
    else:
        centering = back.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        back_issues = back.get('issues', {})
        if not _print_side_issues({k: v for k, v in back_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Grade estimate ────────────────────────────────────────────
    gr   = grade.get('grade_range', '?')
    conf = grade.get('confidence', '?')
    lf   = grade.get('limiting_factor')
    lf_str = f'  [limiting: {lf}]' if lf else ''
    print(f'\n📊  Grade Estimate: {gr}  ({conf} confidence){lf_str}')
    dist = grade.get('distribution', {})
    for g in range(10, 0, -1):
        prob = dist.get(str(g), 0)
        bar  = '█' * int(prob * 32)
        print(f'    PSA {g:2d} │ {bar:<32} {prob*100:5.1f}%')

    # ── Grading decision ──────────────────────────────────────────
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '❓')
    print(f'\n{emoji}  Grading Candidate: {gdc.upper()}')
    print(f'    {gd.get("reason", "")}')

    caveats = gd.get('caveats', [])
    if caveats:
        print(f'\n⛔  Caveats:')
        for c in caveats:
            print(f'    • {c}')

    # ── Combined issues ───────────────────────────────────────────
    print('\n⚠️  Combined Issues (worst-case):')
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'    {cat.upper()}')
            for item in items:
                print(f'      • {item}')
    if not has_any:
        print('    ✓ No significant issues detected')

    # Token usage
    if usage:
        print(f'\n💰  Tokens: {usage.get("input_tokens",0):,} in '
              f'/ {usage.get("output_tokens",0):,} out')
    print(f'\n{sep}\n')


def plot_result(result: dict, label: str = '') -> None:
    """Matplotlib visualisation of a single grading result."""
    grade  = result.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    mode   = result.get('analysis_mode', '?')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: probability bars ────────────────────────────────────
    ax1 = axes[0]
    grades = list(range(1, 11))
    probs  = [dist.get(str(g), 0) * 100 for g in grades]
    colors = [GRADE_COLORS[g] for g in grades]
    bars   = ax1.bar(grades, probs, color=colors, width=0.7, edgecolor='#30363d')
    ax1.set_xlabel('PSA Grade')
    ax1.set_ylabel('Probability (%)')
    ax1.set_title(
        f'{grade.get("grade_range","?")}  ({grade.get("confidence","?")} confidence)',
        color='white')
    ax1.set_xticks(grades)
    ax1.set_ylim(0, max(probs) * 1.15 + 1)
    ax1.spines['bottom'].set_color('#30363d')
    ax1.spines['left'].set_color('#30363d')
    for bar, prob in zip(bars, probs):
        if prob > 2:
            ax1.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.4,
                     f'{prob:.0f}%', ha='center', va='bottom',
                     color='white', fontsize=8)

    # ── Right: summary text ───────────────────────────────────────
    ax2 = axes[1]
    ax2.axis('off')
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '?')

    # Build front/back summary strings
    def side_summary(side: dict, name: str) -> list[str]:
        lines = [f'{name}:']
        if not side.get('assessable', True):
            lines.append('  ✗ not assessable')
            return lines
        c = side.get('centering')
        if c:
            lines.append(f'  centering {c}')
        side_issues = side.get('issues', {})
        count = sum(len(v) for v in side_issues.values() if isinstance(v, list))
        lines.append(f'  {count} issue(s) detected')
        return lines

    lines = [
        f"Card: {card.get('name') or 'Unknown'}",
        f"Set:  {card.get('set') or '?'}  {card.get('year') or ''}",
        f"ID confidence: {card.get('confidence', '?')}",
        f"Mode: {mode}   IQ: {iq.get('status','?')}",
        "",
    ]
    lines += side_summary(front, 'Front')
    lines += side_summary(back, 'Back')
    lines += [
        "",
        f"{emoji} Grading: {gdc.upper()}",
        textwrap.fill(gd.get('reason', ''), 42),
    ]
    caveats = gd.get('caveats', [])
    if caveats:
        lines += ["", "⛔ Caveats:"]
        for c in caveats:
            lines.append(f"  • {textwrap.shorten(c, 40)}")
    lines += ["", "Combined Issues:"]
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            lines.append(f"  {cat.upper()} ({len(items)})")
            for it in items:
                lines.append(f"    • {textwrap.shorten(it, 36)}")
    if not any(issues.get(c) for c in ['centering','corners','edges','surface','other']):
        lines.append('  ✓ No significant issues')

    ax2.text(0.04, 0.97, '\n'.join(lines),
             transform=ax2.transAxes, color='white',
             fontsize=9.5, va='top', family='monospace', linespacing=1.55)
    ax2.set_title(label or 'Assessment', color='white')

    plt.suptitle(label, color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


print('✅ Display helpers ready')

---
## Single Card Test

Paste eBay image URLs (s-l1600) **or** supply local file paths.  
Set `show_prompt=True` to print the exact prompt sent to Claude — useful when debugging prompt changes.

In [ ]:
# ── Configure your test card here ────────────────────────────────────────────

# Option A: eBay image URLs  (replace with real URLs)
TEST_IMAGE_URLS = [
    'https://i.ebayimg.com/images/g/REPLACE_ME/s-l1600.jpg',
    # add more URLs for multi-image tests
]

# Option B: Local files  (comment out TEST_IMAGE_URLS above and use this)
# TEST_LOCAL_PATHS = ['image0_front.jpeg', 'image0_back.jpeg']
TEST_LOCAL_PATHS = None

TEST_TITLE = 'Charizard ex 199/165 Scarlet & Violet 151 — PSA?'
TEST_PRICE = 480.0

SHOW_PROMPT = False   # True → print the full prompt sent to Claude
SHOW_RAW    = False   # True → print Claude's raw string before JSON parse

In [ ]:
# Show thumbnails first
if TEST_LOCAL_PATHS:
    show_images(local_paths=TEST_LOCAL_PATHS)
elif 'REPLACE_ME' not in TEST_IMAGE_URLS[0]:
    show_images(image_urls=TEST_IMAGE_URLS)

# Run grading
print('\nRunning grading pipeline...')
result = grade_with_claude(
    image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
    local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
    title=TEST_TITLE,
    show_prompt=SHOW_PROMPT,
    show_raw=SHOW_RAW,
)

print_result(result, label=TEST_TITLE)

In [ ]:
plot_result(result, label=TEST_TITLE)

In [ ]:
# Inspect the raw parsed JSON (excluding base64 image data)
display_json = {k: v for k, v in result.items() if not k.startswith('_')}
print(json.dumps(display_json, indent=2, ensure_ascii=False))

---
## Consistency Test — Same Card, N Runs

Run the same card multiple times to measure how consistent Claude's output is across calls.
Useful for understanding grade range variance before committing to a prompt.

In [ ]:
N_RUNS = 3   # number of times to repeat the same card

repeat_results = []
for i in tqdm(range(N_RUNS), desc='Consistency runs'):
    r = grade_with_claude(
        image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
        local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
        title=TEST_TITLE,
    )
    r['_run'] = i + 1
    repeat_results.append(r)

# Summary table
rows = []
for r in repeat_results:
    grade = r.get('grade_estimate', {})
    dist  = grade.get('distribution', {})
    gd    = r.get('grading_decision', {})
    rows.append({
        'Run':        r['_run'],
        'Grade Range': grade.get('grade_range', '?'),
        'Confidence':  grade.get('confidence', '?'),
        'Gradable':    gd.get('gradable_candidate', '?'),
        'P(PSA10)':   f"{dist.get('10',0)*100:.1f}%",
        'P(PSA9)':    f"{dist.get('9',0)*100:.1f}%",
        'P(PSA8)':    f"{dist.get('8',0)*100:.1f}%",
        'P(PSA7)':    f"{dist.get('7',0)*100:.1f}%",
        'In$':        r.get('_usage', {}).get('input_tokens', 0),
        'Out$':       r.get('_usage', {}).get('output_tokens', 0),
    })

df_repeat = pd.DataFrame(rows)
print(df_repeat.to_string(index=False))

In [ ]:
# Overlay distributions from all runs
fig, ax = plt.subplots(figsize=(12, 5))

grades = list(range(1, 11))
x = np.arange(len(grades))
width = 0.8 / N_RUNS
cmap = plt.cm.get_cmap('tab10', N_RUNS)

for idx, r in enumerate(repeat_results):
    dist  = r.get('grade_estimate', {}).get('distribution', {})
    probs = [dist.get(str(g), 0) * 100 for g in grades]
    offset = (idx - N_RUNS / 2 + 0.5) * width
    ax.bar(x + offset, probs, width=width * 0.9,
           label=f'Run {idx+1}', color=cmap(idx), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'PSA {g}' for g in grades])
ax.set_ylabel('Probability (%)')
ax.set_title(f'Grade Distribution Consistency — {N_RUNS} runs', color='white')
ax.legend()
ax.spines['bottom'].set_color('#30363d')
ax.spines['left'].set_color('#30363d')
plt.tight_layout()
plt.show()

---
## Batch Test — Multiple Cards

Define a list of test cases below (name, URLs or local paths, title, known grade if available) and run them all in one shot.

In [ ]:
# ── Define test cases ─────────────────────────────────────────────────────────
# Each dict keys:
#   name         : short label for display
#   image_urls   : list of eBay image URLs  (or use local_paths)
#   local_paths  : list of local image paths (comment out image_urls key)
#   title        : listing title (used as identity hint only)
#   price        : listing price in USD
#   known_grade  : actual PSA grade if you have it — used for accuracy calc (optional)

BATCH_CASES = [
    {
        'name':       'Card A',
        'local_paths': ['image0_front.jpeg', 'image0_back.jpeg'],
        'title':       'Card A — test',
        'price':       50.0,
        'known_grade': None,
    },
    {
        'name':       'Card B',
        'local_paths': ['image1_front.jpeg', 'image1_back.jpeg'],
        'title':       'Card B — test',
        'price':       30.0,
        'known_grade': None,
    },
    # Add more cases — paste eBay URLs or local paths
]

In [ ]:
batch_results = []

for i, tc in enumerate(BATCH_CASES):
    print(f"\n[{i+1}/{len(BATCH_CASES)}] {tc['name']}...")
    try:
        r = grade_with_claude(
            image_urls=tc.get('image_urls'),
            local_paths=tc.get('local_paths'),
            title=tc.get('title', ''),
        )
        r['_test_name']   = tc['name']
        r['_price']       = tc.get('price', 0)
        r['_known_grade'] = tc.get('known_grade')
        batch_results.append(r)
        print_result(r, label=tc['name'])
    except Exception as exc:
        print(f'  ❌ Error: {exc}')
        batch_results.append({
            '_test_name': tc['name'],
            '_error': str(exc)
        })

print(f'\n✅ Batch complete: {sum(1 for r in batch_results if "_error" not in r)}/{len(BATCH_CASES)} succeeded')

In [ ]:
# Comparison table
rows = []
for r in batch_results:
    if '_error' in r:
        rows.append({'Name': r['_test_name'], 'Error': r['_error']})
        continue
    grade  = r.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    card   = r.get('card_identity', {})
    gd     = r.get('grading_decision', {})
    iq     = r.get('image_quality', {})
    cv     = r.get('_cv') or {}
    front  = r.get('front_analysis', {})
    back   = r.get('back_analysis', {})
    known  = r.get('_known_grade')

    # Compute expected PSA from distribution
    ev = sum(int(g) * p for g, p in dist.items())

    rows.append({
        'Name':         r.get('_test_name', '?'),
        'Card':         card.get('name') or '?',
        'Mode':         r.get('analysis_mode', '?'),
        'Range':        grade.get('grade_range', '?'),
        'Conf':         grade.get('confidence', '?'),
        'Limit':        grade.get('limiting_factor') or '—',
        'E[PSA]':       f'{ev:.1f}',
        'Known':        known or '—',
        'Gradable':     gd.get('gradable_candidate', '?'),
        'Caveats':      len(gd.get('caveats', [])),
        'F.assess':     '✓' if front.get('assessable') else '✗',
        'B.assess':     '✓' if back.get('assessable') else '✗',
        'IQ':           iq.get('status', '?'),
        'Blur':         f"{cv.get('blur_score', '?')}",
        'Glare%':       f"{cv.get('glare_fraction', 0)*100:.1f}",
        'Issues':       sum(len(r.get('issues', {}).get(c, [])) for c in ['centering','corners','edges','surface','other']),
        'In tok':       r.get('_usage', {}).get('input_tokens', 0),
        'Out tok':      r.get('_usage', {}).get('output_tokens', 0),
    })

df_batch = pd.DataFrame(rows)
print(df_batch.to_string(index=False))

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results to plot')
else:
    n = len(valid)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, valid):
        dist   = r.get('grade_estimate', {}).get('distribution', {})
        probs  = [dist.get(str(g), 0) * 100 for g in range(1, 11)]
        colors = [GRADE_COLORS[g] for g in range(1, 11)]
        bars   = ax.bar(range(1, 11), probs, color=colors, width=0.7, edgecolor='#30363d')
        gr     = r.get('grade_estimate', {}).get('grade_range', '?')
        ax.set_title(f"{r.get('_test_name','?')}\n{gr}", color='white', fontsize=10)
        ax.set_xticks(range(1, 11))
        ax.set_xlabel('PSA Grade')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')
        for bar, prob in zip(bars, probs):
            if prob > 5:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,
                        f'{prob:.0f}%', ha='center', va='bottom',
                        color='white', fontsize=8)

    axes[0].set_ylabel('Probability (%)')
    plt.suptitle('Batch Grade Distributions', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Accuracy Analysis

If you supplied `known_grade` for any test cases, this section measures how well Claude's expected-value estimate matches the actual PSA grade.

In [ ]:
labelled = [
    r for r in batch_results
    if '_error' not in r and r.get('_known_grade') is not None
]

if not labelled:
    print('No labelled test cases (set known_grade in BATCH_CASES to enable accuracy analysis)')
else:
    errors, abs_errors = [], []
    for r in labelled:
        dist  = r.get('grade_estimate', {}).get('distribution', {})
        ev    = sum(int(g) * p for g, p in dist.items())
        known = float(r['_known_grade'])
        err   = ev - known
        errors.append(err)
        abs_errors.append(abs(err))
        print(f"  {r.get('_test_name','?'):20s}  E[PSA]={ev:.1f}  Actual={known:.0f}  Error={err:+.1f}")

    print(f'\n  MAE:  {np.mean(abs_errors):.2f} PSA grades')
    print(f'  Bias: {np.mean(errors):+.2f} (positive = overestimates grade)')
    print(f'  Within ±1 grade: {sum(1 for e in abs_errors if e <= 1)}/{len(abs_errors)}')
    print(f'  Within ±2 grade: {sum(1 for e in abs_errors if e <= 2)}/{len(abs_errors)}')

---
## Issues Frequency Analysis

Aggregates which issue categories appear most often across your batch — useful for spotting systematic bias in the prompt.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']
    counts = {c: 0 for c in cats}
    total_issues = 0
    for r in valid:
        for cat in cats:
            n = len(r.get('issues', {}).get(cat, []))
            counts[cat] += n
            total_issues += n

    fig, ax = plt.subplots(figsize=(9, 4))
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']
    bars = ax.bar(cat_labels, [counts[c] for c in cats],
                  color=bar_colors, edgecolor='#30363d')
    for bar, cat in zip(bars, cats):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                str(counts[cat]),
                ha='center', va='bottom', color='white', fontsize=11)
    ax.set_ylabel('Total issues reported')
    ax.set_title(f'Issue category frequency — {len(valid)} card(s), {total_issues} total issues',
                 color='white')
    ax.spines['bottom'].set_color('#30363d')
    ax.spines['left'].set_color('#30363d')
    plt.tight_layout()
    plt.show()

    print('\nAll reported issues:')
    for cat in cats:
        all_items = []
        for r in valid:
            all_items.extend(r.get('issues', {}).get(cat, []))
        if all_items:
            print(f'\n  {cat.upper()}')
            for item in all_items:
                print(f'    \u2022 {item}')

---
## Side Analysis Summary

Per-side (front vs back) issue frequency breakdown and assessability overview across the batch.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']

    # ── Assessability summary ─────────────────────────────────────
    front_assess = sum(1 for r in valid if r.get('front_analysis', {}).get('assessable', True))
    back_assess  = sum(1 for r in valid if r.get('back_analysis',  {}).get('assessable', True))
    front_only   = sum(1 for r in valid if r.get('analysis_mode') == 'front_only')
    front_back   = sum(1 for r in valid if r.get('analysis_mode') == 'front_back')
    print(f'Cards analysed: {len(valid)}')
    print(f'  front_back mode : {front_back}')
    print(f'  front_only mode : {front_only}')
    print(f'  Front assessable: {front_assess}/{len(valid)}')
    print(f'  Back  assessable: {back_assess}/{len(valid)}')

    # ── Per-side issue counts ─────────────────────────────────────
    front_counts = {c: 0 for c in cats}
    back_counts  = {c: 0 for c in cats}
    for r in valid:
        for cat in cats:
            front_counts[cat] += len(r.get('front_analysis', {}).get('issues', {}).get(cat, []))
            back_counts[cat]  += len(r.get('back_analysis',  {}).get('issues', {}).get(cat, []))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']

    for ax, counts, title in [
        (axes[0], front_counts, 'Front Issues'),
        (axes[1], back_counts,  'Back Issues'),
    ]:
        bars = ax.bar(cat_labels, [counts[c] for c in cats],
                      color=bar_colors, edgecolor='#30363d')
        for bar, cat in zip(bars, cats):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    str(counts[cat]),
                    ha='center', va='bottom', color='white', fontsize=11)
        ax.set_title(title, color='white')
        ax.set_ylabel('Issue count')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')

    plt.suptitle('Front vs Back Issue Breakdown', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Caveats summary ───────────────────────────────────────────
    caveats_total = sum(len(r.get('grading_decision', {}).get('caveats', [])) for r in valid)
    if caveats_total:
        print(f'\n⛔  {caveats_total} caveat(s) across {len(valid)} card(s):')
        for r in valid:
            for c in r.get('grading_decision', {}).get('caveats', []):
                print(f'  [{r.get("_test_name","?")}] {c}')
    else:
        print('\n✓  No caveats — all cards had back images available')